# B04B — Ablation klasifikasi: crop × resolusi
Jalankan setelah cache B04A READY. Pilot default: **VGG16 + DenseNet121 pada CBIS-DDSM**, kandidat
yang dipilih berdasarkan validation B03. Delapan classifier tetap tersedia. EfficientDet tidak ditambahkan
sesuai keputusan terbaru; fokus adalah klasifikasi.

Empat varian memakai sampel, label, split, batch, seed dan protokol training yang sama.
Perbedaan yang diuji adalah crop dan resolusi. Full_224 merupakan kontrol pengulangan B03.
Default 2 model × 4 varian × 1 seed = 8 run. Hasil bukan reproduksi ROI + tabular golden paper.

**Tidak ada cell evaluasi test.** Test B03 pernah dilihat. Hasil B04 adalah validation untuk pengembangan,
bukan bukti peningkatan test terkonfirmasi. ROI lesi, verifikasi identitas pasien INbreast dan grouped
cross-validation merupakan tahap selanjutnya, belum dilakukan notebook ini.

Training B03 V2 dipertahankan: head 5 epoch LR1e-3, fine-tuning maksimal25 LR1e-5, BN frozen non-fused,
class weights dari train, threshold0,5. Pemilihan checkpoint berdasarkan validation macro-F1 lalu recall.
Karena checkpoint dipilih pada validation yang sama, nilainya dapat optimistis.


## Cell 1 — GPU dan Drive


In [ ]:
import os,sys,subprocess,importlib.metadata as metadata
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import tensorflow as tf
tf_minor='.'.join(tf.__version__.split('.')[:2])
try: installed=metadata.version('tf-keras')
except metadata.PackageNotFoundError: installed=''
if '.'.join(installed.split('.')[:2])!=tf_minor:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={tf_minor}.0'])
import tf_keras as keras
gpu=tf.config.list_physical_devices('GPU')
if not gpu:raise RuntimeError('Pilih runtime GPU')
for device in gpu:tf.config.experimental.set_memory_growth(device,True)
keras.mixed_precision.set_global_policy('float32')
tf.config.experimental.enable_op_determinism()
print('TensorFlow',tf.__version__,'TF-Keras',keras.__version__,'GPU',gpu)
from google.colab import drive
drive.mount('/content/drive')


## Cell 2 — Trainer B03 V2 dengan cache terpisah antarvarian


In [ ]:
TRAINER_SOURCE = '\nimport os, io, gc, json, time, hashlib, zipfile, shutil, platform\n\nfrom pathlib import Path\n\nimport numpy as np\n\nimport pandas as pd\n\nimport tensorflow as tf\n\nimport tf_keras as keras\n\nfrom sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,\n    roc_auc_score, average_precision_score, confusion_matrix, balanced_accuracy_score)\n\nREGISTRY = {\n    \'vgg16\': (\'teacher\',\'VGG16\',\'vgg16\'),\n    \'resnet50\': (\'teacher\',\'ResNet50\',\'resnet\'),\n    \'inception_v3\': (\'teacher\',\'InceptionV3\',\'inception_v3\'),\n    \'regnet_y_8gf\': (\'teacher\',\'RegNetY080\',None),\n    \'densenet121\': (\'student\',\'DenseNet121\',\'densenet\'),\n    \'mobilenet_v2\': (\'student\',\'MobileNetV2\',\'mobilenet_v2\'),\n    \'mobilenet_v3\': (\'student\', \'MobileNetV3Large\', None),\n    \'nasnet_mobile\': (\'student\',\'NASNetMobile\',\'nasnet\'),\n}\n\ndef json_safe(value):\n    if isinstance(value, dict): return {str(k):json_safe(v) for k,v in value.items()}\n    if isinstance(value, (tuple,list)): return [json_safe(v) for v in value]\n    if isinstance(value, np.generic): return json_safe(value.item())\n    if isinstance(value, float) and not np.isfinite(value): return None\n    return value\n\ndef save_json(path, obj):\n    path=Path(path); tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(json_safe(obj),indent=2),encoding=\'utf-8\'); os.replace(tmp,path)\n\ndef file_sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef metrics(y, p):\n    y=np.asarray(y,dtype=int); p=np.asarray(p,dtype=float).reshape(-1)\n    if len(y)!=len(p) or not np.isfinite(p).all() or ((p<0)|(p>1)).any():\n        raise ValueError(\'Invalid predictions\')\n    pred=(p>=.5).astype(int)\n    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()\n    return dict(precision=precision_score(y,pred,zero_division=0),\n        recall=recall_score(y,pred,zero_division=0), f1=f1_score(y,pred,zero_division=0),\n        macro_f1=f1_score(y,pred,labels=[0,1],average=\'macro\',zero_division=0),\n        accuracy=accuracy_score(y,pred), balanced_accuracy=balanced_accuracy_score(y,pred),\n        specificity=tn/(tn+fp) if tn+fp else np.nan,\n        roc_auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,\n        average_precision=average_precision_score(y,p) if len(np.unique(y))==2 else np.nan,\n        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),threshold=.5,images=len(y))\n\ndef make_dataset(frame, image_dir, key, batch, seed, epoch=0, training=False, size=224):\n    paths=[str(image_dir/p) for p in frame.cache_member]\n    labels=frame.label.to_numpy(dtype=np.float32)\n    d=tf.data.Dataset.from_tensor_slices((paths,labels))\n    if training:\n        d=d.shuffle(len(frame),seed=seed+epoch,reshuffle_each_iteration=False)\n    preprocess_module=REGISTRY[key][2]\n    def load(index, pair):\n        path,label=pair\n        x=tf.image.decode_png(tf.io.read_file(path),channels=3)\n        x=tf.ensure_shape(tf.cast(x,tf.float32),[size,size,3])\n        if training:\n            # Stateless horizontal flip: image identity is not a laterality target.\n            x=tf.image.stateless_random_flip_left_right(x,seed=tf.stack([\n                tf.cast(seed+epoch*1009,tf.int32),tf.cast(index,tf.int32)]))\n        if preprocess_module:\n            x=getattr(keras.applications,preprocess_module).preprocess_input(x)\n        # MobileNetV3 / RegNet contain their own rescaling and consume [0,255].\n        return x,tf.reshape(label,[1])\n    options=tf.data.Options(); options.experimental_deterministic=True\n    options.threading.private_threadpool_size=2\n    return d.enumerate().map(load,num_parallel_calls=2,deterministic=True).batch(batch).with_options(options).prefetch(1)\n\ndef load_ready_dataset(ready_path, local_root):\n    ready_path=Path(ready_path); meta=json.loads(ready_path.read_text()); folder=ready_path.parent\n    for filename,digest in meta[\'file_sha256\'].items():\n        if file_sha(folder/filename)!=digest: raise RuntimeError(\'B02 cache changed: \'+filename)\n    frames={s:pd.read_csv(folder/(s+\'.csv\'),dtype={\'patient_id\':str,\'sample_id\':str})\n            for s in [\'train\',\'validation\',\'test\']}\n    all_d=pd.concat(frames.values(),ignore_index=True)\n    assert not all_d.sample_id.duplicated().any()\n    assert all_d.groupby(\'patient_id\').split.nunique().max()==1\n    assert all(set(d.label)=={0,1} for d in frames.values())\n    image_dir=Path(local_root)/meta[\'dataset\']/meta[\'file_sha256\'][\'images.zip\'][:16]\n    image_dir.mkdir(parents=True,exist_ok=True)\n    marker=image_dir/\'EXTRACTED.json\'\n    expected=meta[\'file_sha256\'][\'images.zip\']\n    valid=marker.exists() and json.loads(marker.read_text()).get(\'sha256\')==expected\n    if not valid or not all((image_dir/f).is_file() for f in all_d.cache_member):\n        local_zip=image_dir/\'images.zip\'\n        shutil.copyfile(folder/\'images.zip\',local_zip)\n        if file_sha(local_zip)!=expected: raise RuntimeError(\'Local cache copy corrupted\')\n        with zipfile.ZipFile(local_zip) as z:\n            for member in all_d.cache_member:\n                if Path(member).name!=member: raise ValueError(\'Unsafe cache member\')\n                (image_dir/member).write_bytes(z.read(member))\n        save_json(marker,{\'sha256\':expected})\n    # Audit the actual extracted model inputs, including reused local cache.\n    from PIL import Image\n    for row in all_d.itertuples():\n        with Image.open(image_dir/row.cache_member) as im:\n            a=np.asarray(im)\n            if a.shape!=(meta[\'image_size\'],meta[\'image_size\']): raise ValueError(\'Unexpected image size\')\n            if hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:\n                raise RuntimeError(\'Corrupted local image cache\')\n    return frames,image_dir,meta\n\ndef export_predictions(model, frame, ds, out, split, key, dataset, seed):\n    probability=model.predict(ds,verbose=0).reshape(-1)\n    result=metrics(frame.label.to_numpy(),probability)\n    table=frame[[\'sample_id\',\'patient_id\',\'label\']].copy()\n    table[\'probability\']=probability.astype(np.float64)\n    table[\'prediction\']=(probability>=.5).astype(int)\n    table[\'model\'],table[\'dataset\'],table[\'training_seed\']=key,dataset,seed\n    table.to_csv(out/(split+\'_predictions.csv\'),index=False)\n    result.update(model=key,dataset=dataset,training_seed=seed,split=split,role=REGISTRY[key][0])\n    save_json(out/(split+\'_metrics.json\'),result)\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import ConfusionMatrixDisplay\n    fig,ax=plt.subplots(figsize=(5,4.5))\n    names=[\'BI-RADS 1/2\',\'BI-RADS 5/6\'] if dataset==\'inbreast\' else [\'Benign\',\'Malignant\']\n    ConfusionMatrixDisplay(np.array([[result[\'tn\'],result[\'fp\']],[result[\'fn\'],result[\'tp\']]]),display_labels=names).plot(\n        ax=ax,cmap=\'Blues\',values_format=\'d\',im_kw={\'vmin\':0,\'vmax\':int(frame.label.value_counts().max())})\n    ax.set_title(f\'{key} | {dataset}\\n{split}; seed {seed}; threshold 0.5\')\n    fig.tight_layout(); fig.savefig(out/(split+\'_confusion_matrix.png\'),dpi=180);plt.close(fig)\n    return result\n\ndef build_model(key, size=224, seed=42, weights=\'imagenet\', learning_rate=1e-5, stage=\'finetune\'):\n    keras.utils.set_random_seed(seed)\n    _, constructor, _ = REGISTRY[key]\n    base = getattr(keras.applications, constructor)(include_top=False, weights=weights,\n        input_shape=(size,size,3), pooling=\'avg\')\n    base.trainable = stage == \'finetune\'\n    for layer in base.layers:\n        if isinstance(layer, keras.layers.BatchNormalization):\n            layer.trainable = False\n            layer.fused = False\n    inp = keras.Input((size,size,3), name=\'preprocessed_image\')\n    x = base(inp, training=False)\n    x = keras.layers.Dropout(.2, seed=seed, name=\'head_dropout\')(x)\n    output = keras.layers.Dense(1, activation=\'sigmoid\', dtype=\'float32\', name=\'probability\')(x)\n    model = keras.Model(inp, output, name=key)\n    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),\n        loss=keras.losses.BinaryCrossentropy())\n    model.optimizer.build(model.trainable_variables)\n    return model\n\ndef weight_snapshot(model):\n    # Explicitly exclude optimizer: head and fine-tuning have different variable sets.\n    base = next(layer for layer in model.layers if isinstance(layer, keras.Model))\n    return tf.train.Checkpoint(backbone=base, classifier=model.get_layer(\'probability\'))\n\ndef protocol_dir(run_root, dataset, protocol):\n    signature = hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()[:12]\n    return Path(run_root)/dataset/signature\n\ndef run_dir(run_root, dataset, key, protocol):\n    return protocol_dir(run_root,dataset,protocol)/key/f"seed_{protocol[\'training_seed\']}"\n\ndef experiment_config(key, dataset, protocol):\n    return dict(protocol, model=key, dataset=dataset, role=REGISTRY[key][0],\n        constructor=REGISTRY[key][1], preprocessing_module=REGISTRY[key][2])\n\ndef probability_diagnostics(y, p):\n    p=np.asarray(p,dtype=np.float64).reshape(-1)\n    q=np.clip(p,1e-7,1-1e-7); y=np.asarray(y,dtype=np.float64)\n    return dict(probability_min=float(p.min()), probability_max=float(p.max()),\n        probability_std=float(p.std()), positive_prediction_fraction=float((p>=.5).mean()),\n        validation_bce_unweighted=float(-(y*np.log(q)+(1-y)*np.log(1-q)).mean()))\n\ndef preprocessing_audit(frames, image_dir, meta, key, protocol):\n    from PIL import Image\n    subset=frames[\'train\'].iloc[:protocol[\'batch_size\']]\n    raw=np.stack([np.asarray(Image.open(image_dir/name).convert(\'RGB\'),dtype=np.float32)\n                  for name in subset.cache_member])\n    ds=make_dataset(subset,image_dir,key,protocol[\'batch_size\'],protocol[\'training_seed\'],size=meta[\'image_size\'])\n    x,y=next(iter(ds)); actual=x.numpy()\n    if key in [\'vgg16\',\'resnet50\']:\n        expected=raw[...,::-1]-np.array([103.939,116.779,123.68],dtype=np.float32)\n    elif key==\'densenet121\':\n        expected=(raw/255.-np.array([.485,.456,.406],dtype=np.float32))/np.array([.229,.224,.225],dtype=np.float32)\n    elif key in [\'inception_v3\',\'mobilenet_v2\',\'nasnet_mobile\']:\n        expected=raw/127.5-1.\n    else: expected=raw  # RegNet and MobileNetV3 use internal preprocessing.\n    np.testing.assert_allclose(actual,expected,rtol=1e-6,atol=2e-5)\n    np.testing.assert_equal(y.numpy().reshape(-1),subset.label.to_numpy())\n    return dict(model=key,constructor=REGISTRY[key][1],preprocessing=REGISTRY[key][2],\n        checked_split=\'train\',checked_images=len(subset),input_min=float(actual.min()),\n        input_max=float(actual.max()),input_std=float(actual.std()),formula_check=\'passed\')\n\ndef environment_record():\n    return dict(python=platform.python_version(),tensorflow=tf.__version__,tf_keras=keras.__version__,\n        gpu=[tf.config.experimental.get_device_details(g) for g in tf.config.list_physical_devices(\'GPU\')])\n\ndef run_stage(key,frames,image_dir,meta,out,protocol,stage,initial_checkpoint=None):\n    out.mkdir(parents=True,exist_ok=True)\n    complete=out/\'COMPLETE.json\'\n    if complete.exists():\n        result=json.loads((out/\'validation_metrics.json\').read_text())\n        state=json.loads(complete.read_text())\n        return result,state\n    keras.backend.clear_session();gc.collect()\n    head=stage==\'head\'\n    budget=protocol[\'head_epochs\'] if head else protocol[\'finetune_epochs\']\n    learning_rate=protocol[\'head_learning_rate\'] if head else protocol[\'finetune_learning_rate\']\n    latest=tf.train.latest_checkpoint(str(out/\'last\'))\n    model=build_model(key,meta[\'image_size\'],protocol[\'training_seed\'],\n        weights=\'imagenet\' if head and not latest else None, learning_rate=learning_rate,stage=stage)\n    selected_weights=weight_snapshot(model)\n    if not head and not latest:\n        if not initial_checkpoint: raise RuntimeError(\'Fine-tuning requires the selected head-stage weights\')\n        selected_weights.restore(initial_checkpoint).assert_existing_objects_matched()\n    epoch_var=tf.Variable(0,dtype=tf.int64,trainable=False)\n    best_f1=tf.Variable(-1.,dtype=tf.float64,trainable=False)\n    best_recall=tf.Variable(-1.,dtype=tf.float64,trainable=False)\n    wait=tf.Variable(0,dtype=tf.int64,trainable=False)\n    best_epoch=tf.Variable(0,dtype=tf.int64,trainable=False)\n    best_path=tf.Variable(\'\',dtype=tf.string,trainable=False)\n    history_var=tf.Variable(\'[]\',dtype=tf.string,trainable=False)\n    ckpt=tf.train.Checkpoint(model=model,optimizer=model.optimizer,epoch=epoch_var,\n        best_f1=best_f1,best_recall=best_recall,wait=wait,best_epoch=best_epoch,\n        best_path=best_path,history=history_var)\n    manager=tf.train.CheckpointManager(ckpt,str(out/\'last\'),max_to_keep=2)\n    best_manager=tf.train.CheckpointManager(selected_weights,str(out/\'best\'),max_to_keep=2)\n    if latest:\n        ckpt.restore(latest).assert_existing_objects_matched()\n        print(key,stage,\'resume after phase epoch\',int(epoch_var.numpy()),flush=True)\n    train=frames[\'train\']; val=frames[\'validation\']\n    history=json.loads(history_var.numpy().decode())\n    counts=train.label.value_counts().to_dict()\n    weights={int(k):len(train)/(2*v) for k,v in counts.items()}\n    val_ds=make_dataset(val,image_dir,key,protocol[\'batch_size\'],protocol[\'training_seed\'],size=meta[\'image_size\'])\n    if not latest:\n        p0=model.predict(val_ds,verbose=0).reshape(-1)\n        save_json(out/\'initial_validation_diagnostics.json\',dict(metrics(val.label,p0),\n            **probability_diagnostics(val.label,p0),stage=stage,\n            note=\'diagnostic only; not a candidate checkpoint\'))\n    env=environment_record()\n    for epoch in range(int(epoch_var.numpy()),budget):\n        if not head and int(wait.numpy())>=protocol[\'patience\']: break\n        global_epoch=epoch if head else protocol[\'head_epochs\']+epoch\n        train_ds=make_dataset(train,image_dir,key,protocol[\'batch_size\'],protocol[\'training_seed\'],\n            epoch=global_epoch,training=True,size=meta[\'image_size\'])\n        start=time.perf_counter()\n        fit=model.fit(train_ds,epochs=1,class_weight=weights,verbose=2,callbacks=[keras.callbacks.TerminateOnNaN()])\n        train_seconds=time.perf_counter()-start\n        loss=float(fit.history[\'loss\'][0])\n        if not np.isfinite(loss): raise RuntimeError(\'Nonfinite loss; no checkpoint committed\')\n        start=time.perf_counter();p=model.predict(val_ds,verbose=0).reshape(-1)\n        score=metrics(val.label,p)\n        row=dict(epoch=epoch+1,global_epoch=global_epoch+1,stage=stage,loss=loss,\n            learning_rate=learning_rate,train_seconds=train_seconds,validation_seconds=time.perf_counter()-start,\n            gpu=str(env[\'gpu\']),**score,**probability_diagnostics(val.label,p))\n        improved=score[\'macro_f1\']>float(best_f1.numpy()) or (\n            score[\'macro_f1\']==float(best_f1.numpy()) and score[\'recall\']>float(best_recall.numpy()))\n        if improved:\n            best_f1.assign(score[\'macro_f1\']);best_recall.assign(score[\'recall\']);best_epoch.assign(epoch+1);wait.assign(0)\n            best_path.assign(best_manager.save(checkpoint_number=epoch+1))\n        else:wait.assign_add(1)\n        epoch_var.assign(epoch+1);history.append(row);history_var.assign(json.dumps(json_safe(history)))\n        manager.save(checkpoint_number=epoch+1)\n        pd.DataFrame(history).to_csv(out/\'history.csv\',index=False)\n        print(f"{key} {stage} {epoch+1}/{budget}: macro-F1={score[\'macro_f1\']:.4f}, AUC={score[\'roc_auc\']:.4f}, p_std={row[\'probability_std\']:.6f}",flush=True)\n    selected=best_path.numpy().decode()\n    if not selected:raise RuntimeError(\'No completed epoch in this stage\')\n    selected_weights.restore(selected).assert_existing_objects_matched()\n    result=export_predictions(model,val,val_ds,out,\'validation\',key,meta[\'dataset\'],protocol[\'training_seed\'])\n    result.update(best_epoch=int(best_epoch.numpy()),best_stage=stage,parameters=model.count_params(),\n        trainable_parameters=int(sum(np.prod(v.shape) for v in model.trainable_variables)),\n        total_train_minutes=sum(r[\'train_seconds\'] for r in history)/60,epochs_completed=int(epoch_var.numpy()))\n    save_json(out/\'validation_metrics.json\',result)\n    pd.DataFrame(history).to_csv(out/\'history.csv\',index=False)\n    state=dict(best_checkpoint=selected,last_checkpoint=manager.latest_checkpoint,stage=stage,\n        initial_checkpoint=initial_checkpoint,validation_only=True)\n    save_json(complete,state)\n    del model;keras.backend.clear_session();gc.collect()\n    return result,state\n\ndef run_model(key,frames,image_dir,meta,run_root,protocol):\n    out=run_dir(run_root,meta[\'dataset\'],key,protocol);out.mkdir(parents=True,exist_ok=True)\n    config=experiment_config(key,meta[\'dataset\'],protocol)\n    path=out/\'config.json\'\n    if path.exists() and json.loads(path.read_text())!=config:raise RuntimeError(\'Run config mismatch\')\n    save_json(path,config)\n    if (out/\'COMPLETE.json\').exists():\n        print(key,\'already complete under this B04 protocol\')\n        return json.loads((out/\'validation_metrics.json\').read_text()),out\n    save_json(out/\'environment.json\',environment_record())\n    audit=preprocessing_audit(frames,image_dir,meta,key,protocol)\n    save_json(out/\'preprocessing_audit.json\',audit)\n    head,hs=run_stage(key,frames,image_dir,meta,out/\'head\',protocol,\'head\')\n    ft,fs=run_stage(key,frames,image_dir,meta,out/\'finetune\',protocol,\'finetune\',hs[\'best_checkpoint\'])\n    # Global validation selection; retain head-only result if fine-tuning does not improve it.\n    choose_ft=(ft[\'macro_f1\'],ft[\'recall\'])>(head[\'macro_f1\'],head[\'recall\'])\n    selected,state,stage=(ft,fs,\'finetune\') if choose_ft else (head,hs,\'head\')\n    for filename in [\'validation_predictions.csv\',\'validation_confusion_matrix.png\']:\n        shutil.copyfile(out/stage/filename,out/filename)\n    result=dict(selected)\n    result[\'best_phase_epoch\']=selected[\'best_epoch\']\n    result[\'best_epoch\']=selected[\'best_epoch\']+(protocol[\'head_epochs\'] if choose_ft else 0)\n    result[\'epochs_completed\']=head[\'epochs_completed\']+ft[\'epochs_completed\']\n    result[\'total_train_minutes\']=head[\'total_train_minutes\']+ft[\'total_train_minutes\']\n    save_json(out/\'validation_metrics.json\',result)\n    histories=[pd.read_csv(out/s/\'history.csv\') for s in [\'head\',\'finetune\']]\n    pd.concat(histories,ignore_index=True).to_csv(out/\'history.csv\',index=False)\n    save_json(out/\'stage_comparison.json\',dict(head=head,finetune=ft,selected_stage=stage))\n    save_json(out/\'COMPLETE.json\',dict(best_checkpoint=state[\'best_checkpoint\'],best_stage=stage,\n        config_sha256=file_sha(path),validation_only=True))\n    return result,out\n\ndef evaluate_frozen_test(key,frames,image_dir,meta,out,protocol):\n    complete=json.loads((out/\'COMPLETE.json\').read_text())\n    record=dict(config_sha256=file_sha(out/\'config.json\'),test_csv_sha256=meta[\'file_sha256\'][\'test.csv\'],\n        best_checkpoint=complete[\'best_checkpoint\'],threshold=.5,selection=\'validation macro-F1, tie positive recall\')\n    lock=out/\'TEST_LOCK.json\'\n    if lock.exists() and json.loads(lock.read_text())!=record:raise RuntimeError(\'Test protocol changed\')\n    save_json(lock,record)\n    if (out/\'test_metrics.json\').exists():return json.loads((out/\'test_metrics.json\').read_text())\n    keras.backend.clear_session()\n    model=build_model(key,meta[\'image_size\'],protocol[\'training_seed\'],weights=None,\n        learning_rate=protocol[\'finetune_learning_rate\'],stage=\'finetune\')\n    weight_snapshot(model).restore(complete[\'best_checkpoint\']).assert_existing_objects_matched()\n    ds=make_dataset(frames[\'test\'],image_dir,key,protocol[\'batch_size\'],protocol[\'training_seed\'],size=meta[\'image_size\'])\n    result=export_predictions(model,frames[\'test\'],ds,out,\'test\',key,meta[\'dataset\'],protocol[\'training_seed\'])\n    del model;keras.backend.clear_session();gc.collect()\n    return result\n\n'
exec(compile(TRAINER_SOURCE,'<B04_CLASSIFIER_TRAINER>','exec'))
TRAINER_SHA256=hashlib.sha256(TRAINER_SOURCE.encode()).hexdigest()
print('Trainer SHA256:',TRAINER_SHA256)


## Cell 3 — Konfigurasi
Default 8 run. Untuk smoke run gunakan MODELS_TO_RUN=['vgg16'] dan VARIANTS_TO_RUN=['full_224'].
Kemudian perluas tanpa mengganti parameter training. Mulai seed42; tambah seed setelah rancangan tetap.
Jika 512 OOM, gunakan GPU lebih besar atau protokol batch baru yang sama untuk semua pembanding.
Jangan mengurangi batch secara diam-diam untuk satu model/varian.


In [ ]:
RESEARCH=Path('/content/drive/MyDrive/Research')
DATASET='cbis_ddsm'
MODELS_TO_RUN=['vgg16','densenet121']
# MODELS_TO_RUN=list(REGISTRY)  # Setelah pilot ditinjau.
VARIANTS_TO_RUN=['full_224','breast_224','full_512','breast_512']
TRAINING_SEEDS=[42]
BATCH_SIZE=8
HEAD_EPOCHS=5;FINETUNE_EPOCHS=25;PATIENCE=5
HEAD_LEARNING_RATE=1e-3;FINETUNE_LEARNING_RATE=1e-5
B04A_REGISTRY=None  # Tempel path dari B04A jika ada lebih dari satu registry.
ROOT_A=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1'
RUN_ROOT=RESEARCH/'outputs/benchmark_4datasets/b04_ablation_runs_v1'
LOCAL_IMAGES=Path('/content/b04_ablation_images')
if B04A_REGISTRY is None:
    candidates=sorted(ROOT_A.glob('*/registry.json'))
    if len(candidates)!=1:raise RuntimeError('Isi B04A_REGISTRY dengan path B04A. Ditemukan: '+str(candidates))
    B04A_REGISTRY=candidates[0]
READY_REGISTRY=json.loads(Path(B04A_REGISTRY).read_text())
if not MODELS_TO_RUN or not set(MODELS_TO_RUN)<=set(REGISTRY):raise ValueError('Invalid classifier')
if not VARIANTS_TO_RUN or not set(VARIANTS_TO_RUN)<={'full_224','breast_224','full_512','breast_512'}:raise ValueError('Invalid variant')
if not TRAINING_SEEDS or len(set(TRAINING_SEEDS))!=len(TRAINING_SEEDS):raise ValueError('Missing/duplicate seeds')
if any(not isinstance(s,int) or s<0 for s in TRAINING_SEEDS):raise ValueError('Invalid seeds')
if min(BATCH_SIZE,HEAD_EPOCHS,FINETUNE_EPOCHS,PATIENCE)<=0:raise ValueError('Invalid training settings')
for variant in VARIANTS_TO_RUN:
    if READY_REGISTRY.get(DATASET,{}).get(variant,{}).get('status')!='READY':raise RuntimeError('Cache belum READY: '+variant)
print('Planned runs:',len(MODELS_TO_RUN)*len(VARIANTS_TO_RUN)*len(TRAINING_SEEDS))


## Cell 4 — Protokol dan identitas cohort


In [ ]:
def build_protocol(meta,variant,seed):
    return dict(training_seed=seed,split_seed=meta['split_seed'],batch_size=BATCH_SIZE,
        head_epochs=HEAD_EPOCHS,finetune_epochs=FINETUNE_EPOCHS,patience=PATIENCE,
        head_learning_rate=HEAD_LEARNING_RATE,finetune_learning_rate=FINETUNE_LEARNING_RATE,
        image_size=meta['image_size'],pipeline_version=meta['pipeline_version'],variant=variant,
        cohort_sha256=meta['cohort_sha256'],images_sha256=meta['file_sha256']['images.zip'],
        data_file_sha256=meta['file_sha256'],trainer_sha256=TRAINER_SHA256,
        preparation_source_sha256=meta['preprocessing_source_sha256'],
        trainer_patch='b04_two_stage_frozen_bn_nonfused_v1',framework='tf_keras',
        tf_version=tf.__version__,tf_keras_version=keras.__version__,precision='float32',weights='imagenet',
        architecture_registry={k:list(v) for k,v in REGISTRY.items()},
        head='GAP_dropout0.2_sigmoid',batch_norm='frozen',batch_norm_implementation='non_fused',
        augmentation='horizontal_flip',class_weight='balanced_from_train_only',threshold=.5,
        checkpoint_selection='validation macro_f1, tie positive recall; ties across stages retain head',
        fine_tune_start='best head checkpoint, fresh Adam optimizer',head_stopping='fixed epoch budget',
        evaluation='validation_only; previously inspected B03 test excluded from tuning')
cohorts={READY_REGISTRY[DATASET][v]['cohort_sha256'] for v in VARIANTS_TO_RUN}
if len(cohorts)!=1:raise RuntimeError('Ablation cohorts differ')
signature=dict(dataset=DATASET,cohort=next(iter(cohorts)),trainer=TRAINER_SHA256,
    variants=VARIANTS_TO_RUN,models=MODELS_TO_RUN,seeds=TRAINING_SEEDS,
    batch=BATCH_SIZE,head_epochs=HEAD_EPOCHS,finetune_epochs=FINETUNE_EPOCHS,
    head_lr=HEAD_LEARNING_RATE,finetune_lr=FINETUNE_LEARNING_RATE,patience=PATIENCE,
    registry_sha256=file_sha(Path(B04A_REGISTRY)),tensorflow=tf.__version__,tf_keras=keras.__version__)
experiment_id=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()[:12]
REPORT_DIR=RUN_ROOT/'reports'/DATASET/experiment_id
REPORT_DIR.mkdir(parents=True,exist_ok=True)
save_json(REPORT_DIR/'experiment_plan.json',signature)
(REPORT_DIR/'trainer_source.py').write_text(TRAINER_SOURCE,encoding='utf-8')
save_json(REPORT_DIR/'runtime.json',environment_record())


## Cell 5 — Training dan validation
Jika terputus, jalankan ulang dari atas dengan konfigurasi sama. Checkpoint model, optimizer dan status
disimpan per epoch. Run selesai digunakan ulang. Resume tidak diklaim identik bit-per-bit pada semua perangkat.
Perubahan framework/protokol membuat folder berbeda.


In [ ]:
RESULTS=[];COMPLETED_FOLDERS=[];PROTOCOL_FOLDERS=[]
reference_identity=None
for variant in VARIANTS_TO_RUN:
    entry=READY_REGISTRY[DATASET][variant]
    frames,image_dir,meta=load_ready_dataset(Path(entry['path'])/'READY.json',LOCAL_IMAGES)
    identity=pd.concat(frames.values(),ignore_index=True)[['sample_id','patient_id','label','split']].sort_values('sample_id').reset_index(drop=True)
    if reference_identity is None:reference_identity=identity
    else:pd.testing.assert_frame_equal(reference_identity,identity)
    for seed in TRAINING_SEEDS:
        protocol=build_protocol(meta,variant,seed)
        pdir=protocol_dir(RUN_ROOT,DATASET,protocol);pdir.mkdir(parents=True,exist_ok=True)
        save_json(pdir/'protocol.json',protocol);save_json(pdir/'data_provenance.json',meta)
        PROTOCOL_FOLDERS.append(pdir)
        for key in MODELS_TO_RUN:
            print('B04 TRAIN:',DATASET,variant,key,'seed',seed,flush=True)
            result,out=run_model(key,frames,image_dir,meta,RUN_ROOT,protocol)
            RESULTS.append(dict(result,variant=variant,run_folder=str(out),protocol_folder=str(pdir)))
            COMPLETED_FOLDERS.append(out)
            pd.DataFrame(RESULTS).to_csv(REPORT_DIR/'validation_summary.csv',index=False)
    del frames
summary=pd.DataFrame(RESULTS)
display(summary[['variant','model','training_seed','precision','recall','f1','macro_f1','accuracy','roc_auc','best_stage','best_epoch']].round(4))


## Cell 6 — Selisih terhadap kontrol dan diagnosis threshold
Threshold0,5 adalah hasil utama ablation. Youden hanya diagnosis pada validation model terpilih:
optimistis pada validation yang sama, tidak memilih checkpoint dan tidak diterapkan otomatis ke test.
Tidak ada p-value/signifikansi berdasarkan satu atau tiga seed.


In [ ]:
from sklearn.metrics import roc_curve
DIAGNOSTICS=[]
for record in RESULTS:
    f=pd.read_csv(Path(record['run_folder'])/'validation_predictions.csv',float_precision='round_trip')
    y=f.label.to_numpy();p=f.probability.to_numpy()
    fpr,tpr,thresholds=roc_curve(y,p)
    choices=np.where(np.isfinite(thresholds)&(thresholds>=0)&(thresholds<=1))[0]
    if len(choices):
        j=choices[np.argmax((tpr-fpr)[choices])];threshold=float(thresholds[j])
        predicted=(p>=threshold).astype(int)
        DIAGNOSTICS.append(dict(variant=record['variant'],model=record['model'],training_seed=record['training_seed'],
            threshold_youden_validation=threshold,recall_at_05=record['recall'],
            recall_at_youden=recall_score(y,predicted,zero_division=0),
            f1_at_youden=f1_score(y,predicted,zero_division=0),
            note='selected-model validation resubstitution; not independent evaluation'))
pd.DataFrame(DIAGNOSTICS).to_csv(REPORT_DIR/'validation_threshold_diagnostics.csv',index=False)
display(pd.DataFrame(DIAGNOSTICS).round(4))
if 'full_224' in set(summary.variant):
    ms=['precision','recall','f1','macro_f1','accuracy','roc_auc']
    control=summary[summary.variant.eq('full_224')][['model','training_seed']+ms]
    delta=summary.merge(control,on=['model','training_seed'],validate='many_to_one',suffixes=('','_control'))
    for metric in ms:delta['delta_'+metric]=delta[metric]-delta[metric+'_control']
    cols=['variant','model','training_seed']+['delta_'+m for m in ms]
    delta[cols].to_csv(REPORT_DIR/'validation_deltas_vs_full224.csv',index=False)
    display(delta[cols].round(4))
if len(TRAINING_SEEDS)>1:
    aggregate=summary.groupby(['variant','model'])[['precision','recall','f1','macro_f1','accuracy','roc_auc']].agg(['mean','std','count'])
    aggregate.to_csv(REPORT_DIR/'validation_multiseed_descriptive.csv');display(aggregate.round(4))


## Cell 7 — Unduh laporan validation


In [ ]:
report=REPORT_DIR/f'B04B_{DATASET}_{experiment_id}_validation_report.zip'
with zipfile.ZipFile(report,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in set(COMPLETED_FOLDERS):
        for f in folder.rglob('*'):
            if f.is_file() and f.suffix in ['.json','.csv','.png']:
                z.write(f,str(f.relative_to(RUN_ROOT)))
    for folder in set(PROTOCOL_FOLDERS):
        for name in ['protocol.json','data_provenance.json']:
            z.write(folder/name,str((folder/name).relative_to(RUN_ROOT)))
    for f in REPORT_DIR.iterdir():
        if f.is_file() and f.suffix in ['.json','.csv','.py']:z.write(f,'summary/'+f.name)
print(report)
from google.colab import files
files.download(str(report))


## Acuan dan batas verifikasi
Trainer dua tahap dan BN non-fused dipertahankan dari B03 V2. Cache lokal sekarang dibedakan dengan hash
ZIP gambar, agar varian dengan cohort sama tidak tercampur.

Validasi lokal: sintaks notebook, kontrol piksel, audit manifest dan cache dengan data sintetis.
Training aktual memerlukan Google Drive dan GPU Colab; belum dijalankan di lingkungan pembuat notebook.

- https://keras.io/guides/transfer_learning/
- https://doi.org/10.3390/bioengineering12060640 — ROI+tabular berbeda dari eksperimen ini.
- https://github.com/google/automl/tree/master/efficientdet — detektor; dikeluarkan dari kandidat sesuai keputusan terbaru.
